# Full Pipeline: OLS Pairs Trading (Experiment)
**Strategy**: NSE Intraday Pairs Trading — Single-Sided Lagger  
**Stages**:
1. Pearson Correlation Screening
2. Execution Engine (Rolling 20-Day OLS EOD Updates)


In [ ]:
import os, glob, gc, json, shutil
import sqlite3
import pandas as pd
import numpy as np
from scipy.stats import t as t_dist

print("=== /kaggle/input contents ===")
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        fpath = os.path.join(root, f)
        print(f"  {fpath}  ({os.path.getsize(fpath)/(1024**3):.2f} GB)")

hits = glob.glob('/kaggle/input/**/*.sqlite', recursive=True)
if not hits:
    raise FileNotFoundError("No .sqlite found under /kaggle/input")
DB_PATH = hits[0]
print(f"\n✅ DB_PATH = {DB_PATH}")


In [ ]:
print("=== Loading All DB Data ===")
con = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT symbol, timestamp, close FROM ohlcv_1min ORDER BY timestamp", con)
con.close()

df['dt'] = pd.to_datetime(df['timestamp'], unit='s', utc=True).dt.tz_convert('Asia/Kolkata')
time_int = df['dt'].dt.hour * 100 + df['dt'].dt.minute
df_trading = df[(time_int >= 915) & (time_int <= 1529)].copy()
del df; gc.collect()

print("Pivoting to price matrix...")
price_matrix = df_trading.pivot(index='dt', columns='symbol', values='close')
del df_trading; gc.collect()

log_prices = np.log(price_matrix)
print(f"Price matrix: {price_matrix.shape}")


## Stage 1 — Pearson Correlation Screening
**Output**: `pairs_all.csv`, `pairs_top500.csv`


In [ ]:
log_returns = log_prices - log_prices.shift(1)
dates_arr = np.array(price_matrix.index.date)
session_open_mask = np.concatenate([[True], dates_arr[1:] != dates_arr[:-1]])
log_returns.iloc[session_open_mask] = np.nan

print("Computing pairwise Pearson correlation...")
corr_df = log_returns.corr(method='pearson')

symbols = corr_df.columns.tolist()
rows = []
for i in range(len(symbols)):
    for j in range(i + 1, len(symbols)):
        rho = corr_df.iloc[i, j]
        if np.isnan(rho): continue
        n_pair = log_returns[[symbols[i], symbols[j]]].dropna().shape[0]
        if n_pair < 5000: continue
        t_stat = rho * np.sqrt((n_pair - 2) / max(1.0 - rho**2, 1e-12))
        p_val  = 2 * t_dist.sf(abs(t_stat), df=n_pair - 2)
        if p_val >= 0.05: continue
        rows.append({
            "symbol_a": symbols[i], "symbol_b": symbols[j],
            "pearson_rho": round(rho, 6), "t_stat": round(t_stat, 4),
            "p_value": round(p_val, 8), "n_obs": n_pair,
        })

pairs_df = pd.DataFrame(rows).sort_values("pearson_rho", ascending=False).reset_index(drop=True)
pairs_df["rank"] = pairs_df.index + 1

pairs_df.to_csv("pairs_all.csv", index=False)
pairs_df.head(500).to_csv("pairs_top500.csv", index=False)
print("Saved Stage 1 outputs.")

# Define TOP 500 for Stage 3 dynamically
top500 = pairs_df.head(500)
TOP_PAIRS = list(zip(top500["symbol_a"], top500["symbol_b"]))
print(f"\nUsing Top {len(TOP_PAIRS)} Production Pairs for Execution Engine...")


## Stage 3 — Execution Engine (Rolling 20-Day OLS)
**Output**: `ols_production_results.csv`


In [ ]:
ZSCORE_WINDOW = 7500
Z_ENTRY = 2.0
EOD_EXIT_TIME = 1515
BASE_CAPITAL = 10_000.0
LEVERAGE = 5.0
POS_SIZE = BASE_CAPITAL * LEVERAGE
FRICTION_PCT = 0.0005

results_st3 = []

# pre-calculate daily closes
daily_closes = price_matrix.groupby(price_matrix.index.date).last()
dates = daily_closes.index.values

def run_backtest_ols(spread, raw_prices, timestamps):
    spread_s = pd.Series(spread)
    roll_mean = spread_s.rolling(ZSCORE_WINDOW).mean()
    roll_std  = spread_s.rolling(ZSCORE_WINDOW).std()
    z_scores  = ((spread_s - roll_mean) / roll_std.replace(0, np.nan)).values
    time_int = timestamps.hour * 100 + timestamps.minute
    is_eod   = (time_int == EOD_EXIT_TIME)
    cash, pos_qty, pos_type, entry_px = BASE_CAPITAL, 0, 0, 0.0
    trade_log = []
    for t in range(ZSCORE_WINDOW, len(spread)):
        z, price = z_scores[t], raw_prices[t]
        if np.isnan(z) or np.isnan(price) or price <= 0: continue
        if pos_qty > 0:
            if is_eod[t] or (pos_type == 1 and z >= 0) or (pos_type == -1 and z <= 0):
                gross = (price - entry_px) * pos_qty if pos_type == 1 else (entry_px - price) * pos_qty
                net = gross - (pos_qty * price) * FRICTION_PCT
                cash += net
                trade_log.append({"net_pnl": net, "reason": "EOD" if is_eod[t] else "MEAN_REV"})
                pos_qty, pos_type = 0, 0
        if pos_qty == 0 and not is_eod[t]:
            if z <= -Z_ENTRY or z >= Z_ENTRY:
                qty = int(POS_SIZE // price)
                if qty > 0:
                    entry_px, pos_qty = price, qty
                    pos_type = 1 if z <= -Z_ENTRY else -1
                    cash -= (qty * price) * FRICTION_PCT
    total_trades = len(trade_log)
    win_rate = sum(1 for tr in trade_log if tr["net_pnl"] > 0) / total_trades if total_trades > 0 else 0.0
    return cash - BASE_CAPITAL, total_trades, win_rate, sum(1 for tr in trade_log if tr["reason"] == "EOD"), sum(1 for tr in trade_log if tr["reason"] == "MEAN_REV")

for sym_a, sym_b in TOP_PAIRS:
    df_pair = log_prices[[sym_a, sym_b]].dropna(how='any')
    ya, yb, times = df_pair[sym_a].values, df_pair[sym_b].values, df_pair.index
    raw_px = price_matrix[sym_a].loc[times].values
    
    # Calculate OLS spread
    spread = np.full(len(ya), np.nan)
    intraday_dates = times.date
    
    for d_idx in range(20, len(dates)):
        current_date = dates[d_idx]
        prev_20_dates = dates[d_idx-20:d_idx]
        
        y_daily = daily_closes.loc[prev_20_dates, sym_a].values
        x_daily = daily_closes.loc[prev_20_dates, sym_b].values
        
        if np.isnan(y_daily).any() or np.isnan(x_daily).any():
            continue
            
        # OLS: y = alpha + beta * x
        X_mat = np.column_stack([x_daily, np.ones(len(x_daily))])
        try:
            beta, alpha = np.linalg.lstsq(X_mat, y_daily, rcond=None)[0]
        except:
            continue
        
        # Apply stationary parameters to today's 1-min data
        mask = (intraday_dates == current_date)
        spread[mask] = ya[mask] - (alpha + beta * yb[mask])

    # Run backtest
    pnl, trades, win_rate, eod_exits, mean_exits = run_backtest_ols(spread, raw_px, times)
    
    results_st3.append({
        "pair": f"{sym_a}-{sym_b}",
        "ols_net_pnl": pnl,
        "ols_trades": trades,
        "ols_win_rate": round(win_rate, 4),
        "eod_exits": eod_exits,
        "mean_exits": mean_exits
    })

res_df = pd.DataFrame(results_st3)
res_df.to_csv("ols_production_results.csv", index=False)
print("Saved OLS outputs.")
display(res_df)


## Publish Output Dataset


In [ ]:
import json, os, shutil
from kaggle.api.kaggle_api_extended import KaggleApi

os.environ['KAGGLE_USERNAME'] = 'utkarshpatelthefirst'
os.environ['KAGGLE_KEY'] = 'fbef16329099428205f671dd5de8337b'

api = KaggleApi()
api.authenticate()

export_dir = '/kaggle/working/dataset_export'
os.makedirs(export_dir, exist_ok=True)

for f in ['pairs_all.csv', 'pairs_top500.csv', 'ols_production_results.csv']:
    if os.path.exists(f): shutil.copy(f, f'{export_dir}/{f}')

meta = {
    "title"    : "Pairs OLS Production Results v1",
    "id"       : "utkarshpatelthefirst/pairs-ols-production-results-v1",
    "licenses" : [{"name": "CC0-1.0"}]
}
with open(f'{export_dir}/dataset-metadata.json', 'w') as f:
    json.dump(meta, f, indent=2)

try:
    api.dataset_create_new(export_dir, dir_mode='zip', quiet=False)
except Exception as e:
    if "already exists" in str(e).lower() or "409" in str(e):
        print("Dataset exists, updating version...")
        api.dataset_create_version(export_dir, "Update", dir_mode='zip', quiet=False)
    else:
        raise e

print("✅ Dataset ready and published.")
